In [1]:
import sys
from pathlib import Path

package_root = str(Path.cwd().parent)
if package_root not in sys.path:
    sys.path.append(package_root)

In [2]:
from __future__ import annotations
import pandas as pd 
import json
from typing import Dict, List, Tuple, Set
from pathlib import Path
from collections.abc import Iterable
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import time
import re
import numpy as np
from copy import deepcopy
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, KFold
import shutil


In [3]:
pd.set_option("display.max_colwidth", None)

In [4]:
DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
INPUT_FOLDER = DATASET_FOLDER + "/input-data"
DATASET1 = "dataset1"
DATASET2 = "dataset2"
EXTRACTED_DATA_FOLDER = DATASET1 + "/all_traffic.parquet"
DEVICE_TO_BE_INCLUDE = DATASET1 + "/device_to_be_include.json"
EXPORTED_FEATURE_FOLDER = "feature-by-window-size"

In [5]:
Path(EXPORTED_FEATURE_FOLDER).mkdir(parents=True, exist_ok=True)

In [6]:
device_dict_1 = {}
with open(DEVICE_TO_BE_INCLUDE, "r") as f:
    device_dict_1= json.load(f)
device_dict_1

{'d0:52:a8:00:67:5e': {'device_name': 'Smart Things',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'hub',
  'vendor': 'Samsung SmartThings'},
 '44:65:0d:56:cc:d3': {'device_name': 'Amazon Echo',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'smart speaker',
  'vendor': 'Amazon'},
 '70:ee:50:18:34:43': {'device_name': 'Netatmo Welcome',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Netatmo'},
 'f4:f2:6d:93:51:f1': {'device_name': 'TP-Link Day Night Cloud camera',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'TP-Link'},
 '00:16:6c:ab:6b:88': {'device_name': 'Samsung SmartCam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Samsung'},
 '30:8c:fb:2f:e4:b2': {'device_name': 'Dropcam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Dropcam'},
 '00:62:6e:51:27:2e': {'device_name': 'Insteon Camera Wired',
  'ip_address': 

In [7]:
all_data = pd.read_parquet(EXTRACTED_DATA_FOLDER)

In [8]:
all_data.head(5)

,frame.time_epoch,frame.len,eth.src,eth.dst,capture_id
0,1.474553e+09,70,18:b7:9e:02:20:44,14:cc:20:51:33:ea,16-09-23
1,1.474553e+09,66,18:b7:9e:02:20:44,14:cc:20:51:33:ea,16-09-23
2,1.474553e+09,66,14:cc:20:51:33:ea,30:8c:fb:2f:e4:b2,16-09-23
3,1.474553e+09,60,d0:52:a8:00:67:5e,14:cc:20:51:33:ea,16-09-23
4,1.474553e+09,98,14:cc:20:51:33:ea,70:ee:50:18:34:43,16-09-23


In [9]:
KSPLITS = 5
RANDOM_SEED = 42
WINDOW_SECONDS = 60

In [10]:
window_size = [2, 4, 8, 16, 32, 64, 128, 256]
while window_size[-1] < 10800:
    window_size.append(window_size[-1] + 256)

len(window_size)

50

In [11]:
def build_feature(
    raw_df: pd.DataFrame,
    device_to_be_include: set,
    window: int = WINDOW_SECONDS
) -> pd.DataFrame:

    if window <= 0:
        raise ValueError("window must be greater than 0")

    all_features = []

    df = raw_df.copy()

    for capture_id, capture_df in df.groupby(
        "capture_id",
        sort=False,
        observed=True,
    ):

        capture_df = capture_df.copy()

        # Devices appearing as either source or destination
        devices = np.union1d(
            capture_df["eth.src"].dropna().unique(),
            capture_df["eth.dst"].dropna().unique(),
        )

        devices = np.array([
            device
            for device in devices
            if device in device_to_be_include
        ])

        if len(devices) == 0:
            continue

        capture_start = capture_df["frame.time_epoch"].min()

        # ---------------------------------------------------------
        # Assign packets to globally aligned windows
        # ---------------------------------------------------------
        capture_df["window"] = (
            (capture_df["frame.time_epoch"] - capture_start) // window
        ).astype("int32")

        # =========================================================
        # OUTGOING FEATURES
        # =========================================================
        outgoing = (
            capture_df[
                capture_df["eth.src"].isin(devices)
            ]
            .groupby(
                ["window", "eth.src"],
                sort=False,
                observed=True,
            )
            .agg(
                outgoing_packet_count=("frame.len", "size"),
                outgoing_total_bytes=("frame.len", "sum"),
                outgoing_first_timestamp=("frame.time_epoch", "min"),
                outgoing_last_timestamp=("frame.time_epoch", "max"),
            )
            .reset_index()
            .rename(columns={"eth.src": "device"})
        )

        # =========================================================
        # INCOMING FEATURES
        # =========================================================
        incoming = (
            capture_df[
                capture_df["eth.dst"].isin(devices)
            ]
            .groupby(
                ["window", "eth.dst"],
                sort=False,
                observed=True,
            )
            .agg(
                incoming_packet_count=("frame.len", "size"),
                incoming_total_bytes=("frame.len", "sum"),
                incoming_first_timestamp=("frame.time_epoch", "min"),
                incoming_last_timestamp=("frame.time_epoch", "max"),
            )
            .reset_index()
            .rename(columns={"eth.dst": "device"})
        )

        # =========================================================
        # Keep only ACTIVE device-windows
        #
        # outer merge means a row exists when:
        #   - outgoing exists
        #   - incoming exists
        #   - or both exist
        #
        # Completely inactive windows are not created.
        # =========================================================
        features = outgoing.merge(
            incoming,
            on=["window", "device"],
            how="outer",
        )

        features = features.sort_values(
            ["device", "window"]
        ).reset_index(drop=True)

        # ---------------------------------------------------------
        # Missing traffic direction -> 0
        # ---------------------------------------------------------
        traffic_columns = [
            "outgoing_packet_count",
            "outgoing_total_bytes",
            "incoming_packet_count",
            "incoming_total_bytes",
        ]

        features[traffic_columns] = (
            features[traffic_columns].fillna(0)
        )

        # =========================================================
        # AGGREGATED FEATURES
        # =========================================================
        features["aggregated_packet_count"] = (
            features["outgoing_packet_count"]
            + features["incoming_packet_count"]
        )

        features["aggregated_total_bytes"] = (
            features["outgoing_total_bytes"]
            + features["incoming_total_bytes"]
        )

        # =========================================================
        # DEVICE ACTIVITY TIMESTAMPS
        #
        # Sending OR receiving counts as activity.
        # =========================================================

        # First activity during this window
        features["first_active_timestamp"] = (
            features[
                [
                    "outgoing_first_timestamp",
                    "incoming_first_timestamp",
                ]
            ]
            .min(axis=1)
        )

        # Last activity during this window
        features["last_active_timestamp"] = (
            features[
                [
                    "outgoing_last_timestamp",
                    "incoming_last_timestamp",
                ]
            ]
            .max(axis=1)
        )

        # =========================================================
        # IDLE TIME
        #
        # idle_time =
        #     first activity in current active window
        #     -
        #     last activity in previous active window
        #
        # Inactive windows are absent, so the gap naturally spans
        # across them.
        # =========================================================
        features["previous_last_active_timestamp"] = (
            features
            .groupby(
                "device",
                observed=True,
            )["last_active_timestamp"]
            .shift(1)
        )

        features["idle_time"] = (
            features["first_active_timestamp"]
            - features["previous_last_active_timestamp"]
                .fillna(capture_start)
        )

        features["capture_id"] = capture_id

        features = features.rename(
            columns={"device": "mac"}
        )

        # =========================================================
        # Final output
        # =========================================================
        features = features[
            [
                "capture_id",
                "window",
                "mac",

                "outgoing_packet_count",
                "outgoing_total_bytes",

                "incoming_packet_count",
                "incoming_total_bytes",

                "aggregated_packet_count",
                "aggregated_total_bytes",

                "idle_time",
            ]
        ]

        all_features.append(features)

    # =============================================================
    # Empty result
    # =============================================================
    output_columns = [
        "capture_id",
        "window",
        "mac",
        "outgoing_packet_count",
        "outgoing_total_bytes",
        "incoming_packet_count",
        "incoming_total_bytes",
        "aggregated_packet_count",
        "aggregated_total_bytes",
        "idle_time",
    ]

    if not all_features:
        return pd.DataFrame(columns=output_columns)

    result = pd.concat(
        all_features,
        ignore_index=True,
    )

    integer_columns = [
        "outgoing_packet_count",
        "outgoing_total_bytes",
        "incoming_packet_count",
        "incoming_total_bytes",
        "aggregated_packet_count",
        "aggregated_total_bytes",
    ]

    result[integer_columns] = (
        result[integer_columns].astype("int64")
    )

    result["idle_time"] = result["idle_time"].astype("float64")

    return result

In [12]:
def device_capture_repeated_split(
    df: pd.DataFrame,
    set_of_devices_to_be_classified,
    n_splits: int = 5,
    random_state: int = 42,
):
    target_devices = set_of_devices_to_be_classified

    # ---------------------------------------------------------
    # Rows belonging to devices we want to classify
    # ---------------------------------------------------------
    target_mask = df["mac"].isin(target_devices).to_numpy()
    target_positions = np.flatnonzero(target_mask)

    work_df = df.iloc[target_positions][
        ["mac", "capture_id"]
    ].copy()

    # ---------------------------------------------------------
    # Validate devices
    # ---------------------------------------------------------
    existing_devices = set(work_df["mac"].unique())

    missing_devices = target_devices - existing_devices

    if missing_devices:
        raise ValueError(
            f"Devices not found in dataframe: {missing_devices}"
        )

    # Unique (device, capture) pairs
    device_captures = (
        work_df
        .drop_duplicates()
        .reset_index(drop=True)
    )

    capture_counts = (
        device_captures
        .groupby("mac", observed=True)
        .size()
    )

    # A device needs at least 2 captures to have
    # capture-separated train AND validation data.
    insufficient = capture_counts[capture_counts < 2]

    if not insufficient.empty:
        raise ValueError(
            "These devices have fewer than 2 capture_ids, "
            "so capture-level train/validation separation is impossible:\n"
            f"{insufficient}"
        )

    rng = np.random.default_rng(random_state)

    # ---------------------------------------------------------
    # Precompute shuffled capture list for each device
    # ---------------------------------------------------------
    captures_by_device = {}

    for device, group in device_captures.groupby(
        "mac",
        observed=True,
        sort=False,
    ):
        captures = group["capture_id"].to_numpy().copy()
        rng.shuffle(captures)

        captures_by_device[device] = captures

    # ---------------------------------------------------------
    # Build repeated splits
    # ---------------------------------------------------------
    folds = []

    row_devices = work_df["mac"].to_numpy()
    row_captures = work_df["capture_id"].to_numpy()

    for fold_idx in range(n_splits):

        # Which capture is validation for each device
        val_capture_by_device = {
            device: captures[fold_idx % len(captures)]
            for device, captures in captures_by_device.items()
        }

        # Construct validation mask
        val_mask = np.zeros(
            len(work_df),
            dtype=bool,
        )

        for device, val_capture in val_capture_by_device.items():
            val_mask |= (
                (row_devices == device)
                & (row_captures == val_capture)
            )

        folds.append(
            {
                "fold": fold_idx,
                "train_iloc": target_positions[~val_mask],
                "val_iloc": target_positions[val_mask],
                "val_capture_by_device": val_capture_by_device,
            }
        )

    return folds

In [13]:
def numpy_to_json(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.bool_):
        return bool(obj)

    raise TypeError(f"{type(obj)} is not JSON serializable")

In [14]:
set_all_devices = set(device_dict_1.keys())

In [15]:
for window in window_size:
    feature_window = build_feature(raw_df=all_data, device_to_be_include=set_all_devices,window=window)
    training_and_validation_folds = device_capture_repeated_split(df=feature_window, set_of_devices_to_be_classified=set_all_devices, n_splits=KSPLITS,random_state=RANDOM_SEED)

    feature_window_folder = f"{EXPORTED_FEATURE_FOLDER}/{window}"
    data_path = f"{feature_window_folder}/data.parquet"
    training_and_validation_fold_file = f"{feature_window_folder}/training_and_validation_fold.json"

    folder_path = Path(feature_window_folder)
    if folder_path.exists():
        print(f"WARNING: Folder {folder_path.stem} exists, overriding it!")
        shutil.rmtree(folder_path)
    folder_path.mkdir(parents=True, exist_ok=True)

    feature_window.to_parquet(data_path, index=False)
    with open(training_and_validation_fold_file, "w") as f:
        json.dump(
            training_and_validation_folds,
            f,
            indent=4,
            default=numpy_to_json,
        )
    print(f"INFO: Finish processing for window {window}.")




INFO: Finish processing for window 2.
INFO: Finish processing for window 4.
INFO: Finish processing for window 8.
INFO: Finish processing for window 16.
INFO: Finish processing for window 32.
INFO: Finish processing for window 64.
INFO: Finish processing for window 128.
INFO: Finish processing for window 256.
INFO: Finish processing for window 512.
INFO: Finish processing for window 768.
INFO: Finish processing for window 1024.
INFO: Finish processing for window 1280.
INFO: Finish processing for window 1536.
INFO: Finish processing for window 1792.
INFO: Finish processing for window 2048.
INFO: Finish processing for window 2304.
INFO: Finish processing for window 2560.
INFO: Finish processing for window 2816.
INFO: Finish processing for window 3072.
INFO: Finish processing for window 3328.
INFO: Finish processing for window 3584.
INFO: Finish processing for window 3840.
INFO: Finish processing for window 4096.
INFO: Finish processing for window 4352.
INFO: Finish processing for window 